# Hand Open/Closed Detection using MediaPipe

## Objective
This project detects whether a hand is **open or closed** using hand landmarks from MediaPipe.
This project detects whether a hand is **open or closed** using hand landmarks from MediaPipe.

## Idea
- Detect 21 hand landmarks
- Count opened fingers
- Classify the hand:
  - Open 
  - Closed 

## Installation
Install required libraries

In [ ]:
!pip install mediapipe opencv-python

## Import Libraries

In [1]:
import cv2
import mediapipe as mp
import numpy as np
import time

from mediapipe.tasks import python
from mediapipe.tasks.python import vision

## Load Hand Landmarker Model

In [2]:
model_path = 'hand_landmarker.task'

base_options = python.BaseOptions(model_asset_path=model_path)

options = vision.HandLandmarkerOptions(
    base_options=base_options,
    running_mode=vision.RunningMode.VIDEO,
    num_hands=1
)

landmarker = vision.HandLandmarker.create_from_options(options)

print("Model Loaded Successfully!")

Model Loaded Successfully!


## Hand Detection and Classification

### Logic
- Each finger has tip and joint
- If tip is above joint → finger is open
- Otherwise → finger is closed
- Thumb is calculated differently (using x-axis)

### Improvements
- Added thumb detection
- Added smoothing to reduce noise

In [3]:
import cv2
import mediapipe as mp
import time

cap = cv2.VideoCapture(0) # Try 0 or 1 for Iriun Camera

# Hand landmark indices
finger_tips = [8, 12, 16, 20]
finger_pip_joints = [6, 10, 14, 18]

# Stabilization variables
prev_status = "Waiting..."
final_status = "Waiting..."
counter = 0

while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        break

    frame = cv2.flip(frame, 1) # Mirror effect
    h, w, _ = frame.shape

    image_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=image_rgb)

    # Use actual system time to avoid timestamp errors
    timestamp = int(time.time() * 1000)
    results = landmarker.detect_for_video(mp_image, timestamp)

    current_detection = "Waiting..."

    if results.hand_landmarks:
        for hand_landmarks in results.hand_landmarks:
            opened_fingers = 0

            # --- Check four fingers (Index, Middle, Ring, Pinky) ---
            for i in range(4):
                if hand_landmarks[finger_tips[i]].y < hand_landmarks[finger_pip_joints[i]].y:
                    opened_fingers += 1

            # --- Check Thumb ---
            if abs(hand_landmarks[4].x - hand_landmarks[5].x) > 0.05:
                if hand_landmarks[4].y < hand_landmarks[3].y:
                    opened_fingers += 1

            # --- Decision Making ---
            if opened_fingers <= 1:
                current_detection = "Hand Closed"
            elif opened_fingers >= 3:
                current_detection = "Hand Open"
            else:
                current_detection = prev_status

            # --- Set landmark color based on current detection ---
            if current_detection == "Hand Open":
                draw_color = (0, 255, 0) # Green
            elif current_detection == "Hand Closed":
                draw_color = (0, 0, 255) # Red
            else:
                draw_color = (255, 255, 255) # White

            # --- Draw Landmarks ---
            for lm in hand_landmarks:
                cx, cy = int(lm.x * w), int(lm.y * h)
                cv2.circle(frame, (cx, cy), 3, draw_color, -1)

    # --- Stabilization System ---
    if current_detection == prev_status:
        counter += 1
        if counter >= 3:
            final_status = current_detection
    else:
        counter = 0
        prev_status = current_detection

    # --- UI Color Logic ---
    if final_status == "Hand Open":
        color = (0, 255, 0)
    elif final_status == "Hand Closed":
        color = (0, 0, 255)
    else:
        color = (200, 200, 200)

    # --- User Interface (UI) ---
    cv2.rectangle(frame, (0, 0), (350, 80), (40, 40, 40), -1)
    cv2.putText(frame, final_status, (15, 55), 
                cv2.FONT_HERSHEY_SIMPLEX, 1.3, color, 3)

    cv2.imshow('Hand Status Detection', frame)

    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()

## Notes
- Frame is flipped for natural interaction
- MediaPipe uses RGB format
- Each hand has 21 landmarks
- Smoothing is used to reduce flickering
- This can be extended for gesture recognition